In [1]:
import sys
sys.path.append('/Users/mariana/Documents/col/lts4/survan')
sys.path.append('/Users/mariana/Documents/col/lts4/tdsurv/lib')


from tdsurv import CoxPH
from tdsurv import concordance_index as tdsurv_ci
from deep_lambda_cox import DeepLambdaSA

from utils import concordance_index, unroll_time, unroll, score
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt

In [141]:

from sksurv.metrics import concordance_index_censored, integrated_brier_score

In [3]:
config_path = '../configs/config_pbc.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

agent = DeepLambdaSA(config, 42)
seqs = agent.data['seqs'].astype(np.float32)
ts = agent.data['ts']
cs = agent.data['cs']

In [5]:
seqs.shape

(312, 16, 15)

In [7]:
train_gen, test_gen = agent.get_train_test(test_size=.2)

In [9]:
test_gen.X.shape

(62, 16, 15)

In [10]:
_ = agent.train()

In [11]:
seq_val = test_gen.X
ts_val = test_gen.ts
cs_val = test_gen.cs

surv = agent.survival_curve(seq_val)
scores = agent.scores(seq_val, q=0.0)
ci = concordance_index(scores, ts_val, cs_val)
ibs = agent.integrated_brier_score(surv[:,0], ts_val, cs_val)

In [38]:
surv.shape

(62, 16, 17)

In [13]:
ci, ibs

(0.9218604651162791, Array(0.12266017, dtype=float64))

In [15]:
scores.shape

(62,)

In [25]:
ei = (~cs_val)  # Event indicator, the opposite of censoring indicator.

In [27]:
ei != cs_val

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True])

In [30]:
# event_indicator, event_time, estimate

ci_2 = concordance_index_censored(ei, ts_val, scores)[0]

In [31]:
ci_2

0.07418397626112759

In [32]:
ci + ci_2

0.9960444413774067

In [39]:
logits = agent.forward(agent.state.params, seq_val)
log_hs = jax.nn.log_sigmoid(logits)
risk = log_hs[:, 0]

In [173]:
log_hs.shape

(62, 16, 16)

In [62]:
ci_2 = concordance_index_censored(ei, ts_val, risk[:,-1])[0]

In [63]:
ci_2

0.9258160237388724

In [44]:
risk.shape, scores.shape

((62, 16), (62,))

In [60]:
ci = concordance_index(-risk[:, -1], ts_val, cs_val)

In [61]:
ci

0.9218604651162791

In [64]:
concordance_index(np.array([1, 2, 3]), np.array([1, 2, 3]), np.array([0, 0, 0]))

1.0

In [69]:
# event_indicator, event_time, estimate


concordance_index_censored(np.array([1, 1, 1]).astype(bool), np.array([1, 2, 3]), -np.array([1, 2, 3]))

(1.0, 3, 0, 0, 0)

In [205]:
risk.shape

(62, 16)

In [202]:
chunk = 10

r_aux = risk[:, -1][:chunk].round(2)
cs_aux = cs_val[:chunk]
ei_aux = ei[:chunk]
ts_aux = ts_val[:chunk]

In [203]:
concordance_index_censored(ei_aux, ts_aux, r_aux, tied_tol=1e-10)

(0.9714285714285714, 34, 1, 0, 2)

In [204]:
concordance_index(-r_aux, ts_aux, cs_aux), concordance_index(scores[:chunk], ts_aux, cs_aux)

(0.972972972972973, 0.972972972972973)

In [123]:
scores.shape

(62,)

In [128]:
ts.max()

15

In [142]:
# (survival_train, survival_test, estimate, times)[source]
surv.shape

(62, 16, 17)

In [143]:
seq_train = train_gen.X
ts_train = train_gen.ts
cs_train = train_gen.cs

ei_train = (~cs_train)  # Event indicator, the opposite of censoring indicator.

In [162]:
et_train = np.array(list(zip(ei_train, ts_train)), dtype = [('e', bool), ('t', float)])
et_test = np.array(list(zip(ei, ts_val)), dtype = [('e', bool), ('t', float)])

In [182]:
times = np.arange(ts.min(), ts.max()+1)
times = times.astype(float)
times[-1] -= 1e-6

In [183]:
surv[:, 0][:, 1:].shape

(62, 16)

In [184]:
integrated_brier_score(et_train, et_test, surv[:, 0][:, 1:], times)

0.11384898798239258

In [185]:
ibs

Array(0.12266017, dtype=float64)

In [196]:
from functools import partial


def brier_score_numpy(h, surv, ts, cs):
    cs = cs.astype(np.bool_)
    ws = kaplan_meier_numpy(ts - ~cs, ~cs)

    # Sequences that terminated.
    mask = np.where((ts <= h) & ~cs, 1, 0)
    aux = (1/ws[ts-1]) * (0.0 - surv[:, h-1])**2 * mask
    aux = np.where(np.isinf(aux), 0, aux)
    aux = np.where(np.isnan(aux), 0, aux)
    tot = np.sum(aux)

    # Sequences that are still active.
    mask = np.where((ts > h) | ((ts == h) & cs), 1, 0)
    aux = (1 / ws[h - 1]) * (1.0 - surv[:, h - 1]) ** 2 * mask
    aux = np.where(np.isinf(aux), 0, aux)
    aux = np.where(np.isnan(aux), 0, aux)
    aux = np.sum(aux)
    tot += aux
    return tot


def kaplan_meier_numpy(ts, cs):
    """Kaplan-Meier estimator of survival curve."""
    cs = cs.astype(np.bool_)
    steps = np.arange(0, np.max(ts) + 1)
    # Number of individuals known to have survived up to step k = 0, 1, ...
    ns = np.sum(ts[:, np.newaxis] >= steps, axis=0)
    # Number of events that happened at step k = 0, 1, ...
    ds = np.sum(ts[~cs, np.newaxis] == steps, axis=0)
    # Product over k of (1 - empirical hazard at k).
    return np.cumprod(1 - ds / ns)

from functools import partial
import numpy as np

def integrated_brier_score_numpy(surv, ts, cs):
    brier = partial(brier_score_numpy, surv=surv, ts=ts, cs=cs)
    t_max = np.max(ts)
    hs = np.arange(1, t_max + 1)
    
    # Compute Brier scores for each h in hs
    brier_scores = np.array([brier(h) for h in hs])
    
    # Compute the integrated Brier score
    return np.sum(brier_scores) / (t_max * len(ts))

In [197]:
agent.integrated_brier_score(surv[:,0], ts_val, cs_val)

Array(0.12266017, dtype=float64)

In [198]:
type(surv), type(ts_val), type(cs_val)

(jaxlib.xla_extension.ArrayImpl, numpy.ndarray, numpy.ndarray)

In [200]:
integrated_brier_score_numpy(surv[:,0], ts_val, cs_val)

/var/folders/3j/6pbn_5s9411218dlhx_yzrk00000gn/T/ipykernel_46291/4239385805.py:10: RuntimeWarning: divide by zero encountered in divide
  aux = (1/ws[ts-1]) * (0.0 - surv[:, h-1])**2 * mask


0.12266017357121653

In [ ]:
# When evaluating dynamic deep hit: use our concordance index with -risk[:, -1] as the risk score (the hazard risk at the last time step), and
# use our numpy implementation of the integrated Brier score with surv[:, 0] as the survival curve.